In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

raw = Path("Human breast cancer in situ capturing transcriptomics")
out = Path("splits_stratified"); out.mkdir(exist_ok=True)

HER2_MAP = {"HER2_luminal": "HER2+", "HER2_non_luminal": "HER2+",
            "luminal_a": "HER2-", "luminal_b": "HER2-", "tnbc": "HER2-"}

# --- patient-level HER2 table from metadata ---
meta = pd.read_csv(raw / "metadata.csv")
meta.columns = meta.columns.str.strip()
meta["type"] = meta["type"].str.strip().str.lower()
meta["type"] = meta["type"].map({k.lower(): v for k, v in HER2_MAP.items()})
assert meta["type"].notna().all(), "unmapped subtype rows"
meta["patient"] = meta["count_matrix"].str.extract(r"(\d{5})")[0].astype(int)
assert (meta.groupby("patient")["type"].nunique() == 1).all(), "mixed HER2 status within a patient"

patients = meta.groupby("patient")["type"].first().reset_index().rename(columns={"type": "her2"})
her2_of  = dict(zip(patients["patient"], patients["her2"]))

def stratified_split(pats, n_pos, n_neg, seed):
    """Hold out n_pos HER2+ and n_neg HER2- patients from pats (list of ints)."""
    rng = np.random.default_rng(seed)
    pos = [p for p in pats if her2_of[p] == "HER2+"]
    neg = [p for p in pats if her2_of[p] == "HER2-"]
    held_pos = rng.choice(pos, n_pos, replace=False).tolist()
    held_neg = rng.choice(neg, n_neg, replace=False).tolist()
    held = [int(p) for p in held_pos + held_neg]
    rest = [int(p) for p in pats if p not in held]
    return rest, held

def save_split(folder, **sets):
    folder.mkdir(parents=True, exist_ok=True)
    for name, pats in sets.items():
        pd.DataFrame({"patient": pats,
                      "her2": [her2_of[p] for p in pats]}).to_csv(
            folder / f"{name}_patients.csv", index=False)

def check(name, **sets):
    all_p = [p for pats in sets.values() for p in pats]
    assert len(all_p) == len(set(all_p)), f"{name}: patient appears in more than one set"
    for k, pats in sets.items():
        labels = [her2_of[p] for p in pats]
        assert "HER2+" in labels and "HER2-" in labels, f"{name} {k}: single class"
    return {k: (sum(her2_of[p] == "HER2+" for p in v),
                sum(her2_of[p] == "HER2-" for p in v)) for k, v in sets.items()}

all_pats = patients["patient"].tolist()

# --- main split: 2 HER2+ and 3 HER2- held out for validation ---
main_train, main_val = stratified_split(all_pats, n_pos=2, n_neg=3, seed=42)
save_split(out, train=main_train, val=main_val)
print("Main split (HER2+, HER2-):", check("main", train=main_train, val=main_val))

# --- 25 sub-splits of the main train set: 2 HER2+ and 2 HER2- held out as test ---
records = []
for i in range(1, 26):
    seed = 100 + i
    sub_train, sub_test = stratified_split(main_train, n_pos=2, n_neg=2, seed=seed)
    d = out / f"split_{i:02d}"
    save_split(d, train=sub_train, test=sub_test)
    counts = check(f"split_{i:02d}", train=sub_train, test=sub_test)
    records.append(dict(split=f"split_{i:02d}", seed=seed,
                        train_pos=counts["train"][0], train_neg=counts["train"][1],
                        test_pos=counts["test"][0], test_neg=counts["test"][1]))

pd.DataFrame(records).to_csv(out / "split_summary.csv", index=False)
print(pd.DataFrame(records).to_string(index=False))
print("\nAll 25 sub-splits contain both classes in train and test.")